# EDA – Certifications SNEP

**Objectif** : Explorer le dataset des certifications SNEP en vue de construire un graphe de co-artistes et d'appliquer l'algorithme de Louvain pour la détection de communautés musicales.  
**Source** : [snepmusique.com/les-certifications](https://snepmusique.com/les-certifications/)  
**Fichier** : `data/snep_certifications.csv`

## 0. Imports

In [ ]:
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

DATA_PATH = Path('../data/snep_certifications.csv')

## 1. Chargement et aperçu

In [ ]:
df = pd.read_csv(DATA_PATH, sep=';', encoding='utf-8-sig', parse_dates=['Date de sortie', 'Date de constat'], dayfirst=True)
print(f'Shape : {df.shape}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 2. Valeurs manquantes

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Manquants': missing, '% manquants': missing_pct})

## 3. Distribution des certifications

In [ ]:
# Normalisation des niveaux de certification (casse)
df['Certification_norm'] = df['Certification'].str.strip().str.title()

cert_order = ['Or', 'Double Or', 'Platine', 'Double Platine', 'Triple Platine',
              'Diamant', 'Double Diamant', 'Triple Diamant', 'Quadruple Diamant']
cert_counts = df['Certification_norm'].value_counts().reindex(cert_order, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(cert_counts.index, cert_counts.values,
              color=sns.color_palette('YlOrRd', len(cert_counts)))
ax.bar_label(bars, padding=3)
ax.set_title('Distribution des niveaux de certification SNEP')
ax.set_xlabel('Niveau')
ax.set_ylabel('Nombre de certifications')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../plots/certif_distribution.png')
plt.show()

## 4. Distribution des catégories

In [ ]:
# Normalisation : 'Single' -> 'Singles'
df['Categorie_norm'] = df['Categorie'].str.strip().replace('Single', 'Singles')

cat_counts = df['Categorie_norm'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel'))
axes[0].set_title('Répartition par catégorie')

bars = axes[1].barh(cat_counts.index, cat_counts.values,
                     color=sns.color_palette('pastel'))
axes[1].bar_label(bars, padding=3)
axes[1].set_title('Nombre de certifications par catégorie')
axes[1].set_xlabel('Nombre')

plt.tight_layout()
plt.savefig('../plots/categorie_distribution.png')
plt.show()

## 5. Top artistes

In [ ]:
# Exclure les artistes génériques
EXCLUDE = {'COMPILATION', 'MULTI INTERPRETES'}
top_artists = (
    df[~df['Interprete'].isin(EXCLUDE)]['Interprete']
    .value_counts()
    .head(25)
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(x=top_artists.values, y=top_artists.index, ax=ax,
            palette='viridis')
ax.bar_label(ax.containers[0], padding=3)
ax.set_title('Top 25 artistes – nombre de certifications SNEP')
ax.set_xlabel('Nombre de certifications')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../plots/top_artists.png')
plt.show()

In [ ]:
# Top artistes par niveau de certification (diamant uniquement)
diamant_artists = (
    df[df['Certification_norm'].str.contains('Diamant', na=False)]
    .groupby('Interprete')
    .size()
    .sort_values(ascending=False)
    .head(15)
)
print('Top 15 artistes avec le plus de certifications Diamant :')
print(diamant_artists.to_string())

## 6. Analyse temporelle

In [ ]:
df_dated = df.dropna(subset=['Date de sortie']).copy()
df_dated['Annee sortie'] = df_dated['Date de sortie'].dt.year

# Certifications par année de sortie (depuis 2000)
yearly = df_dated[df_dated['Annee sortie'] >= 2000].groupby('Annee sortie').size()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yearly.index, yearly.values, color='steelblue')
ax.set_title('Nombre de certifications SNEP par année de sortie (depuis 2000)')
ax.set_xlabel('Année de sortie')
ax.set_ylabel('Certifications')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../plots/certifs_by_year.png')
plt.show()

In [ ]:
# Durée moyenne d'obtention de la certification (en jours)
df_time = df.dropna(subset=['Date de sortie', 'Date de constat']).copy()
df_time['jours_obtention'] = (df_time['Date de constat'] - df_time['Date de sortie']).dt.days
df_time = df_time[df_time['jours_obtention'] > 0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_time['jours_obtention'] / 365, bins=40, color='coral', edgecolor='white')
ax.axvline(df_time['jours_obtention'].median() / 365, color='navy',
           linestyle='--', label=f"Médiane : {df_time['jours_obtention'].median()/365:.1f} ans")
ax.set_title('Distribution de la durée avant obtention de la certification (années)')
ax.set_xlabel('Durée (années)')
ax.set_ylabel('Fréquence')
ax.legend()
plt.tight_layout()
plt.savefig('../plots/duree_obtention.png')
plt.show()

print(df_time['jours_obtention'].describe())

## 7. Analyse des labels / distributeurs

In [ ]:
def extract_main_label(editeur_str):
    """Extrait le premier label mentionné dans la chaîne éditeur/distributeur."""
    if pd.isna(editeur_str):
        return 'Inconnu'
    return editeur_str.split('/')[0].strip()

df['Label principal'] = df['Editeur / Distributeur'].apply(extract_main_label)

top_labels = df['Label principal'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(x=top_labels.values, y=top_labels.index, ax=ax, palette='crest')
ax.bar_label(ax.containers[0], padding=3)
ax.set_title('Top 20 labels principaux (par nombre de certifications)')
ax.set_xlabel('Certifications')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../plots/top_labels.png')
plt.show()

## 8. Détection des collaborations

Les champs `Interprete` contiennent parfois plusieurs artistes séparés par `FEAT.`, `FEAT`, `&` ou `,`.  
Ces collaborations constituent la matière première du **graphe de co-artistes**.

In [ ]:
COLLAB_PATTERN = re.compile(r'\bFEAT\.?\b|\bFT\.?\b', re.IGNORECASE)
AMP_PATTERN = re.compile(r'\s&\s')

def split_artists(raw: str) -> list[str]:
    """Décompose une chaîne d'artistes en liste d'artistes individuels."""
    # Remplacer les séparateurs de collaboration par un pipe
    s = COLLAB_PATTERN.sub('|', raw)
    s = AMP_PATTERN.sub('|', s)
    artists = [a.strip() for a in s.split('|') if a.strip()]
    return artists

df['Artists_list'] = df['Interprete'].apply(split_artists)
df['N_artistes'] = df['Artists_list'].apply(len)
df['Est_collab'] = df['N_artistes'] > 1

print(f"Entrées solo       : {(~df['Est_collab']).sum()}")
print(f"Entrées collab     : {df['Est_collab'].sum()}")
print(f"Taux collaboration : {df['Est_collab'].mean():.1%}")

In [ ]:
# Artistes les plus souvent en featuring
all_artists_in_collabs = [
    artist
    for artists in df[df['Est_collab']]['Artists_list']
    for artist in artists
]
top_collab_artists = pd.Series(Counter(all_artists_in_collabs)).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=top_collab_artists.values, y=top_collab_artists.index,
            ax=ax, palette='magma_r')
ax.bar_label(ax.containers[0], padding=3)
ax.set_title('Top 20 artistes présents dans des collaborations certifiées')
ax.set_xlabel('Nombre de certifications en collab')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../plots/top_collab_artists.png')
plt.show()

## 9. Construction préliminaire du graphe de co-artistes

On construit un graphe non orienté où :
- **Nœuds** = artistes uniques
- **Arêtes** = paires d'artistes ayant collaboré sur un titre certifié
- **Poids** = nombre de collaborations communes

In [ ]:
from itertools import combinations

edge_counter: Counter = Counter()

for artists in df[df['Est_collab']]['Artists_list']:
    for a, b in combinations(sorted(set(artists)), 2):
        edge_counter[(a, b)] += 1

edges_df = (
    pd.DataFrame.from_records(
        [(a, b, w) for (a, b), w in edge_counter.items()],
        columns=['source', 'target', 'weight']
    )
    .sort_values('weight', ascending=False)
)

print(f'Arêtes totales (paires uniques) : {len(edges_df)}')
print(f'Nœuds distincts                 : {len(set(edges_df.source) | set(edges_df.target))}')
print('\nTop 10 collaborations :')
edges_df.head(10)

In [ ]:
# Distribution du poids des arêtes
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(edges_df['weight'], bins=30, color='teal', edgecolor='white', log=True)
ax.set_title('Distribution du poids des arêtes (nombre de collabs, log)')
ax.set_xlabel('Poids (nombre de titres certifiés en commun)')
ax.set_ylabel('Fréquence (log)')
plt.tight_layout()
plt.savefig('../plots/edge_weight_distribution.png')
plt.show()

In [ ]:
# Visualisation d'un sous-graphe (top 50 arêtes les plus lourdes)
try:
    import networkx as nx
    
    top_edges = edges_df.head(50)
    G_sub = nx.from_pandas_edgelist(top_edges, 'source', 'target', 'weight')
    
    degree = dict(G_sub.degree())
    node_sizes = [degree[n] * 200 for n in G_sub.nodes()]
    
    fig, ax = plt.subplots(figsize=(14, 10))
    pos = nx.spring_layout(G_sub, seed=42, k=1.5)
    nx.draw_networkx_nodes(G_sub, pos, node_size=node_sizes,
                           node_color='steelblue', alpha=0.8, ax=ax)
    nx.draw_networkx_edges(G_sub, pos, alpha=0.4, width=0.8, ax=ax)
    nx.draw_networkx_labels(G_sub, pos, font_size=7, ax=ax)
    ax.set_title('Sous-graphe de co-artistes (top 50 collaborations certifiées SNEP)', fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('../plots/subgraph_top50.png')
    plt.show()
except ImportError:
    print("networkx non installé. Installer avec : pip install networkx")

## 10. Synthèse EDA

### Points clés

| Indicateur | Valeur |
|---|---|
| Certifications totales | 8 384 |
| Catégories principales | Singles (47%), Albums (45%), Vidéos (6%) |
| Niveau majoritaire | Or (57%) |
| Entries avec collaborations | ~1 078 (~13%) |
| Arêtes graphe (paires uniques) | voir cellule 9 |

### Prochaines étapes

1. **Nettoyage** : harmoniser les noms d'artistes (casse, accents, alias), normaliser les séparateurs de featuring
2. **Construction du graphe complet** avec `networkx` ou `igraph`
3. **Louvain** : détecter les communautés (`python-louvain` ou `leidenalg`)
4. **Évaluation** : modularité Q, silhouette score sur embeddings Node2Vec
5. **Recommandation** : étant donné un artiste, retourner les artistes de même communauté triés par centralité ou nombre de certifications